# 09 Model Comparison

Compare saved metrics from trained model notebooks.


## 1. Imports and paths


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

model_root = PROJECT_ROOT / "artifacts" / "models"
print("Project root:", PROJECT_ROOT)
model_root


## 2. Load saved results


In [ ]:
metric_rows = []
subject_rows = []
prediction_rows = []

for model_dir in sorted(model_root.iterdir()):
    if not model_dir.is_dir():
        continue
    metrics_path = model_dir / "test_metrics.json"
    validation_path = model_dir / "validation_metrics.json"
    config_path = model_dir / "model_config.json"
    summary_path = model_dir / "training_summary.json"
    if not metrics_path.exists():
        continue

    with metrics_path.open("r", encoding="utf-8") as handle:
        test_metrics = json.load(handle)
    validation_metrics = {}
    if validation_path.exists():
        with validation_path.open("r", encoding="utf-8") as handle:
            validation_metrics = json.load(handle)
    config = {}
    if config_path.exists():
        with config_path.open("r", encoding="utf-8") as handle:
            config = json.load(handle)
    training_summary = {}
    if summary_path.exists():
        with summary_path.open("r", encoding="utf-8") as handle:
            training_summary = json.load(handle)

    metric_rows.append(
        {
            "model": model_dir.name,
            "validation_macro_f1": validation_metrics.get("macro_f1"),
            "test_macro_f1": test_metrics.get("macro_f1"),
            "weighted_f1": test_metrics.get("weighted_f1"),
            "stress_precision": test_metrics.get("stress_precision"),
            "stress_recall": test_metrics.get("stress_recall"),
            "roc_auc": test_metrics.get("roc_auc"),
            "average_precision": test_metrics.get("average_precision"),
            "parameter_count": config.get("parameter_count"),
            "selected_imbalance_method": config.get("selected_imbalance_method") or training_summary.get("selected_imbalance_method"),
            "best_validation_epoch": config.get("best_validation_epoch") or training_summary.get("best_epoch"),
            "epochs_trained": training_summary.get("epochs_trained"),
            "training_time_seconds": training_summary.get("training_time_seconds"),
            "inference_time_seconds": training_summary.get("inference_time_seconds"),
        }
    )

    per_subject_path = model_dir / "per_subject_metrics.csv"
    if per_subject_path.exists():
        frame = pd.read_csv(per_subject_path)
        frame.insert(0, "model", model_dir.name)
        subject_rows.append(frame)

    prediction_path = model_dir / "test_predictions.csv"
    if prediction_path.exists():
        frame = pd.read_csv(prediction_path)
        frame.insert(0, "model", model_dir.name)
        prediction_rows.append(frame)

comparison = pd.DataFrame(metric_rows).sort_values("validation_macro_f1", ascending=False, na_position="last")
per_subject_comparison = pd.concat(subject_rows, ignore_index=True) if subject_rows else pd.DataFrame()
prediction_comparison = pd.concat(prediction_rows, ignore_index=True) if prediction_rows else pd.DataFrame()
comparison


## 3. Summary charts


In [ ]:
if not comparison.empty:
    axes = comparison.set_index("model")[["validation_macro_f1", "test_macro_f1", "stress_recall", "average_precision"]].plot(
        kind="bar",
        figsize=(10, 4),
        ylim=(0, 1),
    )
    axes.set_ylabel("score")
    axes.set_title("Validation-selected model comparison")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 4. Per-subject comparison


In [ ]:
if not per_subject_comparison.empty:
    subject_pivot = per_subject_comparison.pivot_table(
        index="subject_id",
        columns="model",
        values="macro_f1",
    )
    display(subject_pivot)
    subject_pivot.plot(kind="bar", figsize=(10, 4), ylim=(0, 1))
    plt.ylabel("macro F1")
    plt.title("Per-subject macro F1")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 5. Select best model


In [ ]:
if comparison.empty:
    print("No trained model artifacts found yet.")
else:
    selected_model = comparison.sort_values("validation_macro_f1", ascending=False, na_position="last").iloc[0]
    highest_observed_test = comparison.sort_values("test_macro_f1", ascending=False, na_position="last").iloc[0]
    print(
        f"Validation-selected deployment candidate: {selected_model['model']} "
        f"(validation macro-F1={selected_model['validation_macro_f1']:.3f}, "
        f"test macro-F1={selected_model['test_macro_f1']:.3f})"
    )
    print(
        f"Highest observed test macro-F1: {highest_observed_test['model']} "
        f"({highest_observed_test['test_macro_f1']:.3f})"
    )
    display(comparison)
